In [22]:
from numpy.fft import fft
import numpy as np
from matplotlib import pyplot as plt, ticker
import pandas as pd

# L_p = pivot point to bob's center

Lps = [
  30, 35, 40
]
# we will compare the pendulum at 2.60 forced to the no pendulum at 2.60 forced
motor_num = 17
forcing_freq_hz = 2.60



fft_amplitudes_at_forcing_freqs = []

sr = [240] + [120] * 3

column = ["y"] + ["x"] * 3

n_fft=2**18

files = [f"../data/{motor_num}.txt"] + [f"../data/pendulum_{l}mm_17_motor.txt" for l in Lps]

for f, rate, col in zip(files, sr, column):
        
        df = pd.read_csv(f, sep="\t", skiprows=1)


        df = df.dropna(axis=1, how="all")


        t = df["t"]
        d = df[col]
        mean = np.mean(d)

        # X = fft([v - mean for v in y])
        # N = len(X)
        # n = np.arange(N)
        # T = N/sr
        # freq = n/T
        
        # y = df[w + "y"]
        N = len(d) # use original non padded length
        norm_d = (np.array(d) - np.mean(d))*1000
        # data[w]["norm_y"] = norm_y
        window = np.hanning(N)
        X = fft(norm_d * window, n=n_fft)
        X *= 2 / np.sum(window)   # required bc more samples would make it misleadingly higher. 2 bc of neg freqs# replaces 2 / N when windowed
        n = np.arange(n_fft)
        T = n_fft/rate
        freq = n/T
        # data[w]["freqs"] = freq
        # data[w]["amplitudes"] = np.abs(X)
        
        # N = len(y) # use original non padded length
        # norm_y = (np.array(y) - np.mean(y))*1000
        # window = np.hanning(N)
        # X = fft(norm_y * window, n=n_fft)
        # X *= 2 / np.sum(window)   # required bc more samples would make it misleadingly higher. 2 bc of neg freqs# replaces 2 / N when windowed
        # n = np.arange(n_fft)
        # T = n_fft/sr
        # freq = n/T
        
        idx_of_closest_freq = np.argmin(np.abs(freq - forcing_freq_hz))
        
        fft_amplitude_at_closest_freq = abs(X)[idx_of_closest_freq]
        print("Configuration: ", f)
        print("Ruler Amplitude: ", fft_amplitude_at_closest_freq)
        if len(fft_amplitudes_at_forcing_freqs) > 0:
          original = fft_amplitudes_at_forcing_freqs[0]
          new = fft_amplitude_at_closest_freq
          print("Percent Reduction: ", (original - new ) / original * 100)
        print("\n")
        fft_amplitudes_at_forcing_freqs.append(fft_amplitude_at_closest_freq)



Configuration:  ../data/17.txt
Ruler Amplitude:  29.489264886599216


Configuration:  ../data/pendulum_30mm_17_motor.txt
Ruler Amplitude:  0.05137676851239311
Percent Reduction:  99.82577806293251


Configuration:  ../data/pendulum_35mm_17_motor.txt
Ruler Amplitude:  0.018331465991393285
Percent Reduction:  99.93783681600105


Configuration:  ../data/pendulum_40mm_17_motor.txt
Ruler Amplitude:  0.1681191793678259
Percent Reduction:  99.42989701501776


